# wandb-log-step — ex2: log train + val on the same step using commit=False/True

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `wandb-log-step`. Running the final beacon cell reports progress against the `Logging: wandb.log step` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: wandb.log step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wandb-log-step`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wandb-log-step"
DD_SUBTOPIC = "Logging: wandb.log step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `wandb.log({...}, step=..., commit=...)` — quick refresher

Wandb buffers `log` calls per step. By default each `log` call auto-commits — the buffer is flushed and the step advances. If you need to log train + val metrics on the SAME step, you must tell wandb "don't commit yet" with `commit=False`, then flush with a final `commit=True` (or just omit it on the last call):

```python
wandb.log({'train/loss': train_loss}, step=ex_seen, commit=False)
wandb.log({'val/loss':   val_loss},   step=ex_seen, commit=True)
```

**Why this matters.** Without `commit=False`, the train metric and val metric land on different internal steps even though you passed the same `step=`. Filters comparing train vs val at the same x-axis position break.

**The rule.** All-but-last log on a given step → `commit=False`. Last log on a given step → `commit=True` (the default).

### Exercise 2 — log train + val on the same step using commit=False/True

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `wandb.log(..., commit=False)` for non-final metric dicts and `commit=True` for the final one per step so train + val metrics share a single x-axis position.
> Keywords: wandb, log, commit, train-val, mock
> ```

**KCs targeted:** `wandb-log-step-kwarg`, `wandb-log-commit-semantics`

Implement `ex2_log_train_and_val(train_losses, val_loss, batch_size)`. A fake epoch where every step logs train metrics, and the LAST step ALSO logs a val metric — all on the same step axis as the train metric:

1. Maintain `examples_seen` starting at 0, incrementing by `batch_size` per step.
2. For each `train_loss` at index `i`:
   - `examples_seen += batch_size`.
   - If `i < len(train_losses) - 1` (not the last step):
     Call `wandb.log({'train/loss': train_loss}, step=examples_seen)` (auto-commit is fine; step advances naturally).
   - If `i == len(train_losses) - 1` (last step):
     Call `wandb.log({'train/loss': train_loss}, step=examples_seen, commit=False)` first, THEN call `wandb.log({'val/loss': val_loss}, step=examples_seen, commit=True)` to flush both metrics on the SAME step.
3. Return `examples_seen`.

The test inspects `wandb.log.call_args_list` to verify per-call kwargs.

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def ex2_log_train_and_val(train_losses: list, val_loss: float, batch_size: int) -> int:
    examples_seen = 0
    n = len(train_losses)
    for i, train_loss in enumerate(train_losses):
        examples_seen += batch_size
        if i < n - 1:
            wandb.log({'train/loss': train_loss}, step=examples_seen)
        else:
            wandb.log({'train/loss': train_loss}, step=examples_seen, commit=False)
            wandb.log({'val/loss': val_loss}, step=examples_seen, commit=True)
    return examples_seen


<details><summary>Solution</summary>

```python
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def ex2_log_train_and_val(train_losses: list, val_loss: float, batch_size: int) -> int:
    examples_seen = 0
    n = len(train_losses)
    for i, train_loss in enumerate(train_losses):
        examples_seen += batch_size
        if i < n - 1:
            wandb.log({'train/loss': train_loss}, step=examples_seen)
        else:
            wandb.log({'train/loss': train_loss}, step=examples_seen, commit=False)
            wandb.log({'val/loss': val_loss}, step=examples_seen, commit=True)
    return examples_seen
```

**Why two calls, not one merged dict.** You CAN merge train+val into one log call (`{'train/loss': ..., 'val/loss': ...}`). The two-call commit pattern shines when the val computation happens in a SEPARATE code path (separate eval function) — you don't want to thread the train metric all the way down into the eval function just to flush them together.

**`commit=False` is sticky to the step.** Once you log with `commit=False` on step S, wandb holds the buffer until a `commit=True` (or another `log` call that auto-commits) for step S. Any LATER `log` for the same step lands in the same buffer.

**Real-world variant.** Validation often runs once per epoch, not once per step. Same pattern — `commit=False` on the train tail, then a separate `eval()` call logs val with `commit=True`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()